In [1]:
# Importing standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Optional: prevent TensorFlow from using GPU (for debugging or testing)
#tf.config.set_visible_devices([], 'GPU')

# Check how many GPUs are available
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


In [2]:
import pandas as pd
import zipfile
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile

zip_file = '/content/drive/MyDrive/content/fruits_7_dataset_100x100.zip'

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
  zip_ref.extractall('/content/dataset')

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Normalize pixel values to [0,1] — helps speed up training and stabilize gradients
    shear_range=0.2,        # Apply a slight diagonal transformation (shear) — simulates natural changes in camera angle
    zoom_range=0.2,         # Apply random zoom-in effect — helps the model recognize objects at different scales
    horizontal_flip=True    # Flip images horizontally — helps the model handle symmetry (e.g., cat facing left or right)
)

# Testing data generator — only normalization, no augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

In [5]:
#Collect all fruits data folders to few main category folders
import os
import shutil
from pathlib import Path

base_path = Path('/content/dataset/fruits_7_dataset_100x100/7_fruits')

for folder_type in ['Training', 'Test']:
    current_dir = base_path / folder_type
    if not current_dir.exists():
        continue

    print(f"Starting to organize {folder_type} folder...")

    for subfolder in list(current_dir.iterdir()):
        if subfolder.is_dir():
            main_category = subfolder.name.split()[0].capitalize()

            new_target_dir = current_dir / main_category
            new_target_dir.mkdir(exist_ok=True)


            for img_file in subfolder.glob('*'):
                if img_file.is_file():

                    new_img_name = f"{subfolder.name}_{img_file.name}"
                    shutil.move(str(img_file), str(new_target_dir / new_img_name))

            subfolder.rmdir()

print("Bulk folder restructuring complete! All subcategories are merged.")

Starting to organize Training folder...
Starting to organize Test folder...
🎉 Bulk folder restructuring complete! All subcategories are merged.


In [6]:
training_set = train_datagen.flow_from_directory(
    '/content/dataset/fruits_7_dataset_100x100/7_fruits/Training',
    target_size=(100, 100), # 100 is the size of the DB pictures
    batch_size=32,
    class_mode='categorical'
)

test_set = test_datagen.flow_from_directory(
    '/content/dataset/fruits_7_dataset_100x100/7_fruits/Test',
    target_size=(100, 100),
    batch_size=32,
    class_mode='categorical'
)

Found 33603 images belonging to 8 classes.
Found 11192 images belonging to 8 classes.


In [7]:
print(list(training_set.class_indices.keys()))

['Apple', 'Banana', 'Grape', 'Grapefruit', 'Mandarine', 'Mango', 'Orange', 'Peach']


# New Section

In [8]:
from keras.models import Sequential
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

cnn = Sequential()


cnn.add(Input(shape=(100, 100, 3)))

cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))


cnn.add(Conv2D(filters=64, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))


cnn.add(Conv2D(filters=128, kernel_size=3, activation='relu'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

# Flatten
cnn.add(Flatten())

# Fully Connected
cnn.add(Dense(units=128, activation='relu'))
# 8 units because of 8 Classes
cnn.add(Dense(units=8, activation='softmax'))

In [9]:
test_set.class_indices = training_set.class_indices

cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = cnn.fit(x=training_set, validation_data=test_set, epochs=15)

Epoch 1/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 131s 119ms/step - accuracy: 0.8969 - loss: 0.2904 - val_accuracy: 0.9789 - val_loss: 0.0628
Epoch 2/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 121s 115ms/step - accuracy: 0.9815 - loss: 0.0530 - val_accuracy: 0.9427 - val_loss: 0.2345
Epoch 3/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 145s 119ms/step - accuracy: 0.9896 - loss: 0.0319 - val_accuracy: 0.9724 - val_loss: 0.0899
Epoch 4/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 121s 115ms/step - accuracy: 0.9926 - loss: 0.0232 - val_accuracy: 0.9983 - val_loss: 0.0041
Epoch 5/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 121s 115ms/step - accuracy: 0.9946 - loss: 0.0201 - val_accuracy: 0.9912 - val_loss: 0.0202
Epoch 6/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 121s 115ms/step - accuracy: 0.9939 - loss: 0.0205 - val_accuracy: 0.9971 - val_loss: 0.0073
Epoch 7/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 124s 118ms/step - accuracy: 0.9963 - loss: 0.0123 - val_accuracy: 0.9960 - val_loss: 0.0131
Epoch 8/15
1051/1051 ━━━━━━━━━━━━━━━━━━━━ 125s 119ms/step - ac

In [10]:
cnn.save('fruits_cnn_model.h5')
print("Model created & saved succesfully! ")

Model created & saved succesfully! 


In [11]:
import tensorflow as tf
cnn = tf.keras.models.load_model('fruits_cnn_model.h5')

In [18]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np
import os
from pathlib import Path
from datetime import datetime
import uuid

# Configuration for a clean mobile-friendly layout
st.set_page_config(page_title="Fruit Classifier Mobile", page_icon="📸", layout="centered")

st.title("📸 Mobile Fruit & Vegetable Classifier")
st.write("Take a live photo using your phone camera or upload an image from your gallery.")

# 1. Load and compile the newly trained 8-class model
@st.cache_resource
def load_my_model():
    if os.path.exists('fruits_cnn_model.keras'):
        model_obj = tf.keras.models.load_model('fruits_cnn_model.keras', compile=False)
    else:
        model_obj = tf.keras.models.load_model('fruits_cnn_model.h5', compile=False)

    # Crucial compile for clean probability vectors from Softmax
    model_obj.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model_obj

model = load_my_model()

# Your exact 8 classes mapped perfectly
class_names = ['Apple', 'Banana', 'Grape', 'Grapefruit', 'Mandarine', 'Mango', 'Orange', 'Peach']

# Setup image storage directory
SAVE_DIR = Path("saved_images")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_uploaded_image(file_obj, prefix: str = "mobile") -> Path:
    original_name = getattr(file_obj, "name", "") or ""
    ext = Path(original_name).suffix.lower() if Path(original_name).suffix else ".jpg"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    unique = uuid.uuid4().hex[:8]
    filename = f"{prefix}_{timestamp}_{unique}{ext}"
    out_path = SAVE_DIR / filename
    data = file_obj.getvalue()
    out_path.write_bytes(data)
    return out_path

# 2. Seamless selection tool
source_option = st.selectbox("Choose Input Method:", ("Use Phone Camera", "Upload from Gallery"))

chosen = None

if source_option == "Use Phone Camera":
    # On mobile, this natively triggers your smartphone's built-in camera
    chosen = st.camera_input("Capture a fruit image")
else:
    chosen = st.file_uploader("Select a photo from your gallery", type=["jpg", "jpeg", "png", "webp"])

# 3. Interactive display & inference pipeline
if chosen is not None:
    st.image(chosen, caption="Preview", use_container_width=True)

    # Stacked buttons optimized for vertical single-handed mobile usage
    if st.button("💾 Save Image to Server", use_container_width=True):
        try:
            saved_path = save_uploaded_image(chosen, prefix="phone")
            st.success(f"Saved successfully on server! ✅")
        except Exception as e:
            st.error(f"Failed to save: {e}")

    if st.button("🤖 Run AI Prediction", use_container_width=True):
        with st.spinner('Analyzing your fruit...'):
            try:

                image = Image.open(chosen)
                if image.mode != "RGB":
                    image = image.convert("RGB")


                img_resized = image.resize((100, 100))


                img_array = tf.keras.utils.img_to_array(img_resized)


                img_array = img_array / 255.0


                img_array = np.expand_dims(img_array, axis=0)


                predictions = model.predict(img_array)
                predicted_class_index = np.argmax(predictions)
                confidence = np.max(predictions) * 100


                CONFIDENCE_THRESHOLD = 40.0

                if confidence < CONFIDENCE_THRESHOLD:
                    st.error("❌ **Invalid Object Detected!**")
                    st.info(
                        "The system could not confidently identify this object. "
                        "Supported categories: **Apple, Banana, Grape, Grapefruit, Mandarine, Mango, Orange, or Peach**."
                    )
                else:
                    st.success("🎯 **Analysis Complete!**")
                    st.metric(label="Identified Fruit:", value=class_names[predicted_class_index])
                    st.subheader(f"Confidence Level: {confidence:.2f}%")

            except Exception as e:
                st.error(f"Prediction error: {e}")

Overwriting app.py


In [20]:
!pip install streamlit --quiet
!pip install pyngrok --quiet

from pyngrok import ngrok

NGROK_TOKEN = "ENTER YOUR NGROK API KEY"
ngrok.set_auth_token(NGROK_TOKEN)

#Closing old open con
ngrok.kill()
public_url = ngrok.connect(8501)
print(f"\n🚀 YOUR APP LINK: {public_url} 🚀\n")

# Run strm app
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /dev/null &


🚀 YOUR APP LINK: NgrokTunnel: "https://fledgier-rancidly-milly.ngrok-free.dev" -> "http://localhost:8501" 🚀



2026-05-24 11:43:33.585 Uvicorn server started on 0.0.0.0:8501
2026-05-24 11:43:51.256544: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779623031.258142   36793 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13121 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-05-24 11:43:58.980 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-05-24 11:44:03.177 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
I0000 00:00: